# 04 Service frequency from GTFS
Turns the England GTFS timetable into **bus departures per hour by time band** for each LSOA.

Method: pick one representative day, count departures at every stop in three time bands (morning 07-10, off-peak 10-16, evening 16-19), join stops to LSOAs, divide by the hours in each band.

Outputs (in one pass over the timetable):
- `lsoa_frequency.parquet`: a representative **weekday** (feeds the score, clustering and allocation);
- `lsoa_frequency_weekend.parquet`: a **Saturday** and a **Sunday**, for the weekday-versus-weekend view in the dashboard;
- `lsoa_frequency_snapshot2.parquet` (optional): a weekday from a **later download of the same feed**, to track service change. Put the second download in `data/raw/gtfs_england_snapshot2.zip` and set `DATES2`.
- `gtfs_dates.csv`: the dates used.

In [1]:
from pathlib import Path
import zipfile
import numpy as np, pandas as pd, geopandas as gpd
RAW = Path("../data/raw"); CLEAN = Path("../data/clean")
GTFS = RAW / "gtfs_england.zip"
DATES = {"weekday": "2026-10-06", "saturday": "2026-10-10", "sunday": "2026-10-11"}   # Tuesday, Saturday, Sunday inside the feed
GTFS2 = RAW / "gtfs_england_snapshot2.zip"      # optional second download of the feed (service-change tracking)
DATES2 = {"weekday": None}                      # e.g. {"weekday": "2026-12-08"}: a Tuesday inside the second feed
BANDS = {"morning": (7, 10), "off_peak": (10, 16), "evening": (16, 19)}   # [start, end) hour
CHUNK = 5_000_000
print(GTFS.exists(), GTFS.stat().st_size / 1e9 if GTFS.exists() else "")

True 1.162992616


## 1. Functions

In [2]:
def active_trips(z, date):
    """Trip ids that run on `date`, from calendar.txt (weekly pattern, date range) and calendar_dates.txt (added/removed days)."""
    d = pd.Timestamp(date); ds = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    cal = pd.read_csv(z.open("calendar.txt"), dtype={"service_id": str, "start_date": str, "end_date": str})
    services = set(cal[(cal[dow] == 1) & (cal.start_date <= ds) & (cal.end_date >= ds)].service_id)
    if "calendar_dates.txt" in z.namelist():
        cd = pd.read_csv(z.open("calendar_dates.txt"), dtype={"service_id": str, "date": str})
        cd = cd[cd.date == ds]
        services |= set(cd[cd.exception_type == 1].service_id)
        services -= set(cd[cd.exception_type == 2].service_id)
    trips = pd.read_csv(z.open("trips.txt"), usecols=["trip_id", "service_id"], dtype=str)
    return set(trips[trips.service_id.isin(services)].trip_id)

def departures_by_stop(z, active):
    """One pass over stop_times.txt. `active` = {key: set of trip ids}. Returns {key: DataFrame(stop_id, morning, off_peak, evening)}."""
    union = set().union(*active.values())
    counts = {k: {b: {} for b in BANDS} for k in active}
    rows_read = 0
    for chunk in pd.read_csv(z.open("stop_times.txt"), usecols=["trip_id", "stop_id", "departure_time"], dtype=str, chunksize=CHUNK):
        rows_read += len(chunk)
        chunk = chunk[chunk.trip_id.isin(union)]
        hour = pd.to_numeric(chunk.departure_time.str.slice(0, 2), errors="coerce") % 24
        for k, trips in active.items():
            keep = chunk.trip_id.isin(trips)
            for b, (a, e) in BANDS.items():
                s = chunk.loc[keep & (hour >= a) & (hour < e), "stop_id"].value_counts()
                for stop, v in s.items():
                    counts[k][b][stop] = counts[k][b].get(stop, 0) + int(v)
        print(f"\rrows read: {rows_read:,}", end="")
    out = {}
    for k in active:
        dep = pd.DataFrame(counts[k]).reindex(columns=list(BANDS)).fillna(0).astype(int)
        dep.index.name = "stop_id"; out[k] = dep.reset_index()
    return out

def lsoa_frequency(z, dep, lsoa, base):
    """Departures per hour per LSOA (all stops inside it). LSOAs with no service get 0."""
    stops = pd.read_csv(z.open("stops.txt"), usecols=["stop_id", "stop_lat", "stop_lon"], dtype={"stop_id": str}).merge(dep, on="stop_id", how="inner")
    sg = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs=4326).to_crs(27700)
    j = gpd.sjoin(sg, lsoa, how="inner", predicate="within")
    f = j.groupby("LSOA21CD")[list(BANDS)].sum()
    for b, (a, e) in BANDS.items():
        f[f"freq_{b}"] = f[b] / (e - a)
    f = f[[f"freq_{b}" for b in BANDS]].reset_index()
    return base.merge(f, on="LSOA21CD", how="left").fillna(0)

## 2. Trips that run on each date

In [3]:
z = zipfile.ZipFile(GTFS)
cal = pd.read_csv(z.open("calendar.txt"), dtype=str); print("feed validity:", cal.start_date.min(), "to", cal.end_date.max())
active = {k: active_trips(z, d) for k, d in DATES.items()}
for k, v in active.items():
    print(f"{k:9s} {DATES[k]} ({pd.Timestamp(DATES[k]).day_name()}): {len(v):,} active trips")
assert len(active["weekday"]) > 0, "No trips on the weekday: change DATES inside the feed validity"
active = {k: v for k, v in active.items() if len(v) > 0}      # a weekend date outside the feed is skipped with a warning
print("days with data:", list(active))

feed validity: 20260930 to 20271201


weekday   2026-10-06 (Tuesday): 384,291 active trips
saturday  2026-10-10 (Saturday): 336,639 active trips
sunday    2026-10-11 (Sunday): 211,326 active trips
days with data: ['weekday', 'saturday', 'sunday']


## 3. Count departures per stop and time band (one pass)

In [4]:
dep = departures_by_stop(z, active)
dep["weekday"].describe()

rows read: 5,000,000

rows read: 10,000,000

rows read: 15,000,000

rows read: 20,000,000

rows read: 25,000,000

rows read: 30,000,000

... (2 outputs left out)

rows read: 45,000,000

rows read: 50,000,000

rows read: 52,609,308

,morning,off_peak,evening
count,249320.000000,249320.000000,249320.000000
mean,11.012305,22.710601,10.797609
std,16.588016,33.245858,16.494746
min,0.000000,0.000000,0.000000
25%,2.000000,6.000000,2.000000
50%,6.000000,12.000000,6.000000
75%,13.000000,26.000000,12.000000
max,328.000000,639.000000,323.000000


## 4. Stops to LSOAs, departures per hour

In [5]:
lsoa = gpd.read_file(RAW / "lsoa_boundaries_2021_bsc.geojson")
lsoa = lsoa[lsoa["LSOA21CD"].str.startswith("E")].to_crs(27700)[["LSOA21CD", "geometry"]]
base = pd.read_parquet(CLEAN / "lsoa_base.parquet")[["LSOA21CD"]]
freq = {k: lsoa_frequency(z, dep[k], lsoa, base) for k in dep}
freq["weekday"].describe().T

,count,mean,std,min,25%,50%,75%,max
freq_morning,33755.0,27.018180,43.048350,0.0,6.333333,15.333333,32.000000,2064.000000
freq_off_peak,33755.0,27.856540,43.434268,0.0,6.500000,16.000000,33.166667,1947.333333
freq_evening,33755.0,26.488471,42.805066,0.0,6.000000,15.000000,31.333333,2056.666667


## 5. Checks and save

In [6]:
f = freq["weekday"]
print("LSOAs with no morning service (weekday):", (f.freq_morning == 0).sum(), "of", len(f))
print(f[[c for c in f if c.startswith("freq_")]].sum().round(0))
f.to_parquet(CLEAN / "lsoa_frequency.parquet", index=False)

wk = base.copy()
for k, tag in [("saturday", "sat"), ("sunday", "sun")]:
    if k in freq:
        g = freq[k].rename(columns={f"freq_{b}": f"freq_{tag}_{b}" for b in BANDS})
        g[f"freq_{tag}_mean"] = g[[f"freq_{tag}_{b}" for b in BANDS]].mean(axis=1)
        wk = wk.merge(g, on="LSOA21CD")
wk.to_parquet(CLEAN / "lsoa_frequency_weekend.parquet", index=False)
dates = [{"snapshot": 1, "day": k, "date": DATES[k], "active_trips": len(active[k])} for k in active]
print({k: round(float(freq[k][[f"freq_{b}" for b in BANDS]].mean(axis=1).mean()), 2) for k in freq})

LSOAs with no morning service (weekday): 2510 of 33755
freq_morning     911999.0
freq_off_peak    940298.0
freq_evening     894118.0
dtype: float64
{'weekday': 27.12, 'saturday': 23.4, 'sunday': 13.49}


## 6. Optional: second snapshot (service change over time)
Download the England timetable feed again at a later date, save it as `data/raw/gtfs_england_snapshot2.zip`, set `DATES2["weekday"]` to a Tuesday inside it and re-run this notebook.

In [7]:
if GTFS2.exists() and DATES2.get("weekday"):
    z2 = zipfile.ZipFile(GTFS2)
    a2 = {"weekday": active_trips(z2, DATES2["weekday"])}
    assert len(a2["weekday"]) > 0, "No trips on DATES2['weekday'] in the second feed"
    f2 = lsoa_frequency(z2, departures_by_stop(z2, a2)["weekday"], lsoa, base)
    f2.to_parquet(CLEAN / "lsoa_frequency_snapshot2.parquet", index=False)
    dates.append({"snapshot": 2, "day": "weekday", "date": DATES2["weekday"], "active_trips": len(a2["weekday"])})
    print("snapshot 2 saved")
else:
    print("No second snapshot (optional). See the note above.")
pd.DataFrame(dates).to_csv(CLEAN / "gtfs_dates.csv", index=False); pd.DataFrame(dates)

No second snapshot (optional). See the note above.


,snapshot,day,date,active_trips
0,1,weekday,2026-10-06,384291
1,1,saturday,2026-10-10,336639
2,1,sunday,2026-10-11,211326
